In [ ]:
import pandas as pd

In [ ]:
covid_df = pd.read_csv("italy-covid-daywise.csv")
covid_df.head()

In [ ]:
covid_df.info()

In [ ]:
covid_df.describe()

In [ ]:
covid_df.columns

In [ ]:
covid_df.shape

In [ ]:
import jovian

## Retrieving data from a data frame

In [ ]:
# Pandas format is similar to this
covid_data_dict = {
    'date': ['2020-08-30', '2020-08-31', '2020-09-01', '2020-09-02', '2020-09-03'],
    'new_cases': [1444, 1365, 996,  975, 1326],
    'new_death': [1, 4, 6, 8, 6],
    'new_tests': [53541, 42583, 54395, None, None]
}

In [ ]:
# Pandas format is not similar to this
covid_data_list = [
    {'date': '2020-09-01', 'new_cases': 996, 'new_deaths': 6, 'new_tests': None},
    {'date': '2020-09-02', 'new_cases': 996, 'new_deaths': 6, 'new_tests': 3564},
    {'date': '2020-09-03', 'new_cases': 996, 'new_deaths': 6, 'new_tests': 77459},
]

In [ ]:
covid_data_dict['new_cases']

In [ ]:
covid_df['new_cases'].head()

In [ ]:
type(covid_df['new_cases'])

Just like arrays, you can retrieve a specific value with a series using the indexing
notation []

In [ ]:
covid_df['new_cases'][45]

In [ ]:
covid_df['new_tests'][26]

Pandas also provides the .at method to directly retrieve data at a specific row and column

In [ ]:
covid_df.at[35, 'new_cases']

In [ ]:
covid_df.at[20, 'new_tests']

In [ ]:
type(covid_df.at[15, 'new_deaths'])

Instead of using the indexing notation [], Pandas also allow accessing columns as properties
of the data frame using the `.` notation. However, this method only works for columns whose 
names do not contain special characters

In [ ]:
covid_df.new_cases.head()

Further, you can also pass a list of columns within the indexing notation [] to access a subset of the
data frame with just the given columns

In [ ]:
cases_df = covid_df[['date', 'new_cases']]
cases_df.head()

Note, however that the new data frame `cases_df` is simply a 'view' of the original data frame `covid_df`.
Sometimes you might need a full copy of the data frame, in which case you use the `copy` method

In [ ]:
covid_df_copy = covid_df.copy()

The data within `covid_df_copy` is completely separat from covid_df, and changing values inside one of them 
will not affect the other

To access a specific row of data, Pandas provides the `.loc` method

In [ ]:
covid_df.loc[13]

Each retrieved row is also a Series object

In [ ]:
type(covid_df.loc[13])

In [ ]:
covid_df.head(5)

In [ ]:
covid_df.tail(10)

In [ ]:
covid_df.at[0, 'new_tests']

In [ ]:
type(covid_df.at[0, 'new_tests'])

The distinction between 0 and NaN is subtle but important. In this dataset, it represents that daily test numbers
were not reported on specific dates

We can find the first index that doesn't contain an NaN values using `first_valid_index` method

In [ ]:
covid_df.new_tests.first_valid_index()

Let's look at a few rows before and after the index to verify that the values indeed change from NaN to actual
numbers. We can do this by passing a range to loc

In [ ]:
covid_df.loc[13 : 16]

The .sample method can be used to retrieve a random sample of rows from the data frame

In [ ]:
covid_df.sample(10)

Notice that even though we have taken a random sample, the original index of each row has been preserve. This is an
important and useful property of data frames - each row of data has an index associated with it.

# Analyzing Data from data frames

Q: What is the total number of reported cases and deaths related to Covid-19 in Italy

Similar to Numpy arrays, a Pandas series supports the `sum` method to answer these questions

In [ ]:
total_cases = covid_df.new_cases.sum()
total_deaths = covid_df.new_deaths.sum()

print("The number of reported cases is {} and the number of reported deaths is {}".format(total_cases, total_deaths))

Q: What is the overall death rate (ratio of reported deaths to reported cases)

In [ ]:
death_rate = covid_df.new_deaths.sum() / covid_df.new_cases.sum()

print("The overall reported death rate in Italy is {:.2f}%".format(death_rate))

Q: What is the overall number of tests conducted? A total of 935310 tests were conducted before daily test numbers were being reported

In [ ]:
initial_tests = 935310
total_tests = initial_tests + covid_df.new_cases.sum()
total_tests

Q: What fraction of tests returned a positive result?

In [ ]:
positive_rate = total_cases / total_deaths

print("{:.2f}% of tests in Italy led to a positive diagnosis.".format(positive_rate))

# Querying and Sorting rows
Let's say we want only to look at the days which had more than 1000 reported cases

In [ ]:
high_new_cases = covid_df.new_cases > 1000

In [ ]:
high_new_cases.head()

In [ ]:
covid_df[high_new_cases].head()

We can write this succintly on a single line by passing the boolean expression as an index to the data frame

In [ ]:
high_cases_df = covid_df[covid_df.new_cases > 1000]
high_cases_df.head()

We can also formulate more complex queries that involve multiple columns. As an example, let's try to determine the days when the ration of cases reported to tests conducted is higher than the overall `positive_rate`

In [ ]:
positive_rate
high_ratio_df = covid_df[covid_df.new_cases / covid_df.new_tests > positive_rate]
high_ratio_df.head()

Perfoming operations on multiple columns results in a new series

In [ ]:
covid_df.new_cases / covid_df.new_tests

Further, we can use this series to add a new column to the data frame

In [ ]:
covid_df['positive_rate'] = covid_df.new_cases / covid_df.new_tests
covid_df.head()

For now let's remove the `positive_rate` column using the  `drop` method

In [ ]:
covid_df.drop(columns=['positive_rate'], inplace=True)

In [ ]:
covid_df.head()

# Sorting rows using column values

The rows can also be created by a specific column using `.sort_values`. Let's sort to identify the days with the highest number of cases, then chain it with the `head` method to get the 10 days with the most cases

In [ ]:
# ascending=False sorts the values in decreasing order
covid_df.sort_values('new_cases', ascending=False).head(10)

It looks like the last two weeks of March had the highest number of daily cases. Let's compare this to the days where the highest number of deaths were recorded

In [ ]:
covid_df.sort_values('new_deaths', ascending=False).head(10)

Let's also look at the days with the least number of cases


In [ ]:
# by default ascending=True meaning values will be sortede in ascending order
covid_df.sort_values('new_cases').head(10)

Seems like the count of new cases on June 20th was -148, a negative number! Not something we might have expected, but that's the nature of real world data. It could simpy be a data entry error, or it's possible that the government may have issued to account for miscounting in the past

Let's look at some days before and after June 20th

In [ ]:
covid_df.loc[36 : 40]

If this was indeed a data entry error, we can use one of the following approaches for dealing with the missing or faulty value
    1. Replace with 0
    2. Replace it with the average of the entire column
    3. Replace it with the average values on the previous and next date
    4. Discard the row entirely

The .at method can be used to modify a specific value within the data frame

In [ ]:
covid_df.at[38, 'new_cases'] = (covid_df.at[37, 'new_cases'] + covid_df.at[39, 'new_cases']) / 2

In [ ]:
covid_df.loc[36 : 40]